In [1]:
from pathlib import Path
import os, sys

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

%load_ext autoreload
%autoreload 2

print("Working directory:", Path.cwd())

Working directory: C:\Users\makar\Python\auto-data-scientist


In [2]:
# Docker check
import subprocess
print(subprocess.run(["docker", "ps"], capture_output=True, text=True).stdout)

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES



In [3]:
# LLM check — also tells you if you're still rate-limited from before
from src.llm import chat
print(chat([{"role": "user", "content": "Reply with the single word: ready"}]))

ready


In [4]:
import os
for f in ["src/sandbox.py", "src/llm.py", "src/profiler.py", "src/schemas.py",
         "src/agents/planner.py", "src/agents/coder.py", "src/agents/reviewer.py",
         "src/orchestrator.py", "src/tracking.py", "src/report.py"]:
    print(f, "->", "OK" if os.path.exists(f) else "MISSING")

src/sandbox.py -> OK
src/llm.py -> OK
src/profiler.py -> OK
src/schemas.py -> OK
src/agents/planner.py -> OK
src/agents/coder.py -> OK
src/agents/reviewer.py -> OK
src/orchestrator.py -> OK
src/tracking.py -> OK
src/report.py -> OK


In [5]:
import pickle
from src.config import RUNS_DIR

ws4 = RUNS_DIR / "part4_full_run_v4" / "workspace"   # adjust if you used a different folder
with open(ws4 / "run_result.pkl", "rb") as f:
    result_final = pickle.load(f)

print("Overall success:", result_final.success)
for log in result_final.logs:
    print(f"Step {log.step_id} ({log.title}): approved={log.approved}")

Overall success: False
Step 1 (Data Exploration): approved=True
Step 2 (Missing Values & Encoding): approved=True
Step 3 (Feature Engineering / Selection): approved=False


In [6]:
from src.agents.coder import solve_step
from src.profiler import quick_profile

profile = quick_profile(ws4 / "data.csv")
step3_task = result_final.plan.steps[2].task   # Step 3

outcome = solve_step(step3_task, profile, ws4, max_attempts=1, verbose=True)
print("Coder success:", outcome.success)

[coder] attempt 1/1: OK
Coder success: True


In [ ]:
print("--- STDOUT ---")
print(outcome.result.stdout)
print("--- STDERR ---")
print(outcome.result.stderr)

In [ ]:
from src.agents.reviewer import review_step

if outcome.success:
    review = review_step(step3_task, outcome.code, outcome.result.stdout)
    print("Approved:", review.approved)
    print("Reason:", review.reason)
    print("Leakage flags:", review.leakage_flags)
else:
    print("Coder failed — nothing to review. See stderr:", outcome.result.stderr)

In [ ]:
print(outcome.code)

In [ ]:
import pandas as pd
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.pipeline import Pipeline

df = pd.read_csv(ws4 / "data.csv")
y = df["target"]
X = df.drop(columns=["target"])

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("rfe", RFE(estimator=LogisticRegression(max_iter=1000, solver="liblinear"),
               n_features_to_select=20, step=1)),
    ("clf", LogisticRegression(max_iter=1000, solver="liblinear"))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y, cv=cv, scoring="accuracy")
print("Cross-validated accuracy: %.4f ± %.4f" % (scores.mean(), scores.std()))

In [ ]:
from src.orchestrator import run_pipeline
from src.config import RUNS_DIR
import pickle

ws5 = RUNS_DIR / "part4_full_run_v5" / "workspace"
ws5.mkdir(parents=True, exist_ok=True)

goal = "Predict whether a tumor is malignant or benign, using the 'target' column."

result_v5 = run_pipeline(
    csv_path=RUNS_DIR / "part4_full_run" / "source_data.csv",
    goal=goal,
    workspace=ws5,
)

print("\nOverall success:", result_v5.success)

# Save immediately -- don't reuse this variable name for anything else later
with open(ws5 / "run_result.pkl", "wb") as f:
    pickle.dump(result_v5, f)
print("Saved result_v5 to disk.")

In [ ]:
from src.tracking import log_run_to_mlflow
from src.report import generate_report

run_id = log_run_to_mlflow(result_v5, ws5, goal)
print("Logged to MLflow, run_id:", run_id)

report_path = generate_report(result_v5, goal, ws5, ws5 / "report.md")
print("Report saved to:", report_path)

In [ ]:
from IPython.display import Markdown, display

with open(ws5 / "report.md", encoding="utf-8") as f:
    content = f.read()
display(Markdown(content))

In [ ]:
from src.report import generate_report

report_path = generate_report(result_v5, goal, ws5, ws5 / "report_v2.md")
print("Report saved to:", report_path)

from IPython.display import Markdown, display
with open(report_path, encoding="utf-8") as f:
    display(Markdown(f.read()))

In [ ]:
from src.orchestrator import run_pipeline
from src.config import RUNS_DIR
import pickle

ws6 = RUNS_DIR / "part4_full_run_v6" / "workspace"
ws6.mkdir(parents=True, exist_ok=True)

result_v6 = run_pipeline(
    csv_path=RUNS_DIR / "part4_full_run" / "source_data.csv",
    goal=goal,
    workspace=ws6,
)

print("Overall success:", result_v6.success)

with open(ws6 / "run_result.pkl", "wb") as f:
    pickle.dump(result_v6, f)
print("Saved result_v6 to disk.")

In [ ]:
for log in result_v6.logs:
    print(f"Step {log.step_id}: {len(log.review_history)} review attempt(s)")

In [ ]:
with open("src/orchestrator.py", encoding="utf-8") as f:
    content = f.read()
print("review_history" in content)

In [6]:
from pathlib import Path
import os, sys

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

%load_ext autoreload
%autoreload 2

from src.orchestrator import run_pipeline
from src.config import RUNS_DIR
import pickle

ws7 = RUNS_DIR / "part4_full_run_v7" / "workspace"
ws7.mkdir(parents=True, exist_ok=True)
goal = "Predict whether a tumor is malignant or benign, using the 'target' column."

result_v7 = run_pipeline(
    csv_path=RUNS_DIR / "part4_full_run" / "source_data.csv",
    goal=goal,
    workspace=ws7,
)

print("Overall success:", result_v7.success)

with open(ws7 / "run_result.pkl", "wb") as f:
    pickle.dump(result_v7, f)

for log in result_v7.logs:
    print(f"Step {log.step_id}: {len(log.review_history)} review attempt(s)")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


SyntaxError: invalid syntax (orchestrator.py, line 1)

In [1]:
   from pathlib import Path
   import os, sys

   if Path.cwd().name == "notebooks":
       os.chdir(Path.cwd().parent)
   sys.path.insert(0, str(Path.cwd()))

   %load_ext autoreload
   %autoreload 2

In [2]:
   with open("src/orchestrator.py", encoding="utf-8") as f:
       first_line = f.readline()
   print(repr(first_line))

'%%writefile src/orchestrator.py\n'


In [3]:
%%writefile src/orchestrator.py
from dataclasses import dataclass, field
from pathlib import Path

from src.agents.coder import solve_step
from src.agents.planner import make_plan
from src.agents.reviewer import review_step
from src.profiler import quick_profile


@dataclass
class StepLog:
    """A record of what happened on one plan step, for the final report."""
    step_id: int
    title: str
    task: str
    coder_success: bool
    coder_attempts: int
    approved: bool
    review_reason: str
    leakage_flags: list
    code: str
    stdout: str
    stderr: str
    review_history: list = field(default_factory=list)


@dataclass
class RunResult:
    """Everything about one full pipeline run, from goal to finish (or failure)."""
    success: bool
    plan: object                 # the Plan object from the planner
    logs: list = field(default_factory=list)
    failed_at_step: int = None   # None if the whole plan completed


def _files_in(workspace: Path) -> list:
    """List files currently in the workspace, so we can tell the coder what already exists."""
    return sorted(p.name for p in workspace.iterdir() if p.is_file())


def run_pipeline(csv_path, goal: str, workspace: Path,
                 max_step_retries: int = 2, coder_attempts_per_try: int = 4,
                 verbose: bool = True) -> RunResult:
    """Run the full agent pipeline: plan the goal, then execute and review each step."""
    workspace = Path(workspace)
    workspace.mkdir(parents=True, exist_ok=True)

    import shutil
    shutil.copy(csv_path, workspace / "data.csv")

    profile = quick_profile(workspace / "data.csv")

    if verbose:
        print("[orchestrator] Planning...")
    plan = make_plan(goal, profile)
    if verbose:
        print(f"[orchestrator] Plan has {len(plan.steps)} steps, problem_type={plan.problem_type}")

    logs = []

    for step in plan.steps:
        if verbose:
            print(f"\n[orchestrator] --- Step {step.id}: {step.title} ---")

        existing_files = [f for f in _files_in(workspace) if f != "step.py"]
        task_with_context = (
            f"{step.task}\n\n"
            f"IMPORTANT: If you save any processed/transformed dataset to CSV, you MUST include "
            f"the target column (keep its original name) alongside the features — later steps "
            f"will need to load labels from these files, not just the feature values.\n\n"
            f"Files already present in the working directory from earlier steps "
            f"(read/reuse them as needed, e.g. via pd.read_csv or joblib.load): {existing_files}"
        )

        review = None
        outcome = None
        review_history = []

        for retry in range(max_step_retries + 1):
            outcome = solve_step(task_with_context, profile, workspace,
                                max_attempts=coder_attempts_per_try, verbose=verbose)

            if not outcome.success:
                break

            review = review_step(step.task, outcome.code, outcome.result.stdout)
            review_history.append({
                "attempt": retry + 1,
                "approved": review.approved,
                "reason": review.reason,
            })

            if review.approved:
                break

            if verbose:
                print(f"[orchestrator] Reviewer rejected (retry {retry + 1}/{max_step_retries}): "
                      f"{review.reason}")
            task_with_context = (
                f"{step.task}\n\n"
                f"IMPORTANT: If you save any processed/transformed dataset to CSV, you MUST include "
                f"the target column (keep its original name) alongside the features — later steps "
                f"will need to load labels from these files, not just the feature values.\n\n"
                f"Your previous attempt was rejected by a reviewer for this reason: {review.reason}\n"
                f"Leakage/quality flags: {review.leakage_flags}\n"
                f"Write a corrected version that avoids this problem.\n\n"
                f"Files already present in the working directory: {existing_files}"
            )

        log = StepLog(
            step_id=step.id, title=step.title, task=step.task,
            coder_success=outcome.success, coder_attempts=outcome.attempts,
            approved=bool(review and review.approved),
            review_reason=review.reason if review else "Coder never produced working code.",
            leakage_flags=review.leakage_flags if review else [],
            code=outcome.code, stdout=outcome.result.stdout,
            stderr=outcome.result.stderr,
            review_history=review_history,
        )
        logs.append(log)

        if not log.approved:
            if verbose:
                print(f"[orchestrator] Step {step.id} FAILED. Stopping pipeline.")
            return RunResult(success=False, plan=plan, logs=logs, failed_at_step=step.id)

    if verbose:
        print("\n[orchestrator] All steps completed and approved.")
    return RunResult(success=True, plan=plan, logs=logs)

Overwriting src/orchestrator.py


In [4]:
with open("src/orchestrator.py", encoding="utf-8") as f:
    first_line = f.readline()
print(repr(first_line))

'from dataclasses import dataclass, field\n'


In [5]:
from src.orchestrator import run_pipeline
from src.config import RUNS_DIR
import pickle

ws7 = RUNS_DIR / "part4_full_run_v7" / "workspace"
ws7.mkdir(parents=True, exist_ok=True)
goal = "Predict whether a tumor is malignant or benign, using the 'target' column."

result_v7 = run_pipeline(
    csv_path=RUNS_DIR / "part4_full_run" / "source_data.csv",
    goal=goal,
    workspace=ws7,
)

print("Overall success:", result_v7.success)

with open(ws7 / "run_result.pkl", "wb") as f:
    pickle.dump(result_v7, f)
print("Saved result_v7 to disk.")

[orchestrator] Planning...
[orchestrator] Plan has 6 steps, problem_type=classification

[orchestrator] --- Step 1: Data Exploration ---
[coder] attempt 1/4: OK

[orchestrator] --- Step 2: Missing Values & Encoding ---
[coder] attempt 1/4: OK

[orchestrator] --- Step 3: Feature Engineering/Selection ---
[coder] attempt 1/4: OK

[orchestrator] --- Step 4: Model Training & Comparison ---
[coder] attempt 1/4: OK

[orchestrator] --- Step 5: Evaluation & Visualization ---
[coder] attempt 1/4: OK

[orchestrator] --- Step 6: Model Persistence ---
[coder] attempt 1/4: OK
[orchestrator] Reviewer rejected (retry 1/2): The code fits the final model on `data.csv`, which likely contains the entire dataset (including test data), causing data leakage; the task asked to fit on the full training data only.
[coder] attempt 1/4: OK

[orchestrator] All steps completed and approved.
Overall success: True
Saved result_v7 to disk.


In [6]:
for log in result_v7.logs:
    print(f"Step {log.step_id}: {len(log.review_history)} review attempt(s)")
    for entry in log.review_history:
        print(f"   attempt {entry['attempt']}: approved={entry['approved']}")

Step 1: 1 review attempt(s)
   attempt 1: approved=True
Step 2: 1 review attempt(s)
   attempt 1: approved=True
Step 3: 1 review attempt(s)
   attempt 1: approved=True
Step 4: 1 review attempt(s)
   attempt 1: approved=True
Step 5: 1 review attempt(s)
   attempt 1: approved=True
Step 6: 2 review attempt(s)
   attempt 1: approved=False
   attempt 2: approved=True


In [7]:
from src.report import generate_report

report_path = generate_report(result_v7, goal, ws7, ws7 / "report.md")
print("Report saved to:", report_path)

from IPython.display import Markdown, display
with open(report_path, encoding="utf-8") as f:
    display(Markdown(f.read()))

SyntaxError: invalid syntax (report.py, line 1)

In [8]:
from src.report import generate_report

report_path = generate_report(result_v7, goal, ws7, ws7 / "report.md")
print("Report saved to:", report_path)

from IPython.display import Markdown, display
with open(report_path, encoding="utf-8") as f:
    display(Markdown(f.read()))

SyntaxError: invalid syntax (report.py, line 1)

In [9]:
%%writefile src/report.py
from pathlib import Path

from src.orchestrator import RunResult


def generate_report(result: RunResult, goal: str, workspace: Path, output_path: Path) -> Path:
    """Turn a RunResult into one readable Markdown file."""
    lines = [
        f"# Auto Data Scientist Report",
        f"",
        f"**Goal:** {goal}",
        f"**Problem type:** {result.plan.problem_type}",
        f"**Target column:** {result.plan.target_column}",
        f"**Overall result:** {'SUCCESS' if result.success else f'FAILED at step {result.failed_at_step}'}",
        f"",
        f"## Steps",
    ]

    for log in result.logs:
        status = "PASSED" if log.approved else "FAILED"
        lines += [
            f"",
            f"### Step {log.step_id}: {log.title} — {status}",
            f"",
            f"**Task:** {log.task}",
            f"",
            f"**Coder attempts:** {log.coder_attempts}",
            f"**Reviewer verdict:** {log.review_reason}",
        ]
        if log.leakage_flags:
            lines.append(f"**Leakage flags:** {', '.join(log.leakage_flags)}")

        if len(log.review_history) > 1:
            lines += ["", "**Review history:**", ""]
            for entry in log.review_history:
                verdict = "APPROVED" if entry["approved"] else "REJECTED"
                lines.append(f"- Attempt {entry['attempt']}: {verdict} — {entry['reason']}")

        if log.stdout.strip():
            lines += ["", "**Output:**", "```", log.stdout.strip(), "```"]

    charts = sorted(Path(workspace).glob("*.png"))
    if charts:
        lines += ["", "## Charts", ""]
        for chart in charts:
            lines.append(f"![{chart.stem}]({chart.name})")

    output_path = Path(output_path)
    output_path.write_text("\n".join(lines), encoding="utf-8")
    return output_path

Overwriting src/report.py


In [10]:
with open("src/report.py", encoding="utf-8") as f:
    first_line = f.readline()
print(repr(first_line))

'from pathlib import Path\n'


In [11]:
for f in ["src/sandbox.py", "src/llm.py", "src/profiler.py", "src/schemas.py",
         "src/agents/planner.py", "src/agents/coder.py", "src/agents/reviewer.py",
         "src/orchestrator.py", "src/tracking.py", "src/report.py"]:
    with open(f, encoding="utf-8") as file:
        first = file.readline()
    status = "BROKEN" if first.strip().startswith("%%writefile") else "OK"
    print(f"{f:30} {status}")

src/sandbox.py                 OK
src/llm.py                     OK
src/profiler.py                OK
src/schemas.py                 OK
src/agents/planner.py          OK
src/agents/coder.py            OK
src/agents/reviewer.py         OK
src/orchestrator.py            OK
src/tracking.py                OK
src/report.py                  OK


In [12]:
from src.report import generate_report

report_path = generate_report(result_v7, goal, ws7, ws7 / "report.md")
print("Report saved to:", report_path)

from IPython.display import Markdown, display
with open(report_path, encoding="utf-8") as f:
    display(Markdown(f.read()))

Report saved to: C:\Users\makar\Python\auto-data-scientist\runs\part4_full_run_v7\workspace\report.md


# Auto Data Scientist Report

**Goal:** Predict whether a tumor is malignant or benign, using the 'target' column.
**Problem type:** classification
**Target column:** target
**Overall result:** SUCCESS

## Steps

### Step 1: Data Exploration — PASSED

**Task:** Load the dataset, compute summary statistics, visualize distributions of each feature, and examine class balance of the target.

**Coder attempts:** 1
**Reviewer verdict:** The code correctly loads the data, prints summary statistics, visualizes feature distributions, and reports class balance as requested.

**Output:**
```
Shape: (569, 31)
First 5 rows:
    mean radius  mean texture  ...  worst fractal dimension  target
0        17.99         10.38  ...                  0.11890       0
1        20.57         17.77  ...                  0.08902       0
2        19.69         21.25  ...                  0.08758       0
3        11.42         20.38  ...                  0.17300       0
4        20.29         14.34  ...                  0.07678       0

[5 rows x 31 columns]

Summary statistics:
        mean radius  mean texture  ...  worst fractal dimension      target
count   569.000000    569.000000  ...               569.000000  569.000000
mean     14.127292     19.289649  ...                 0.083946    0.627417
std       3.524049      4.301036  ...                 0.018061    0.483918
min       6.981000      9.710000  ...                 0.055040    0.000000
25%      11.700000     16.170000  ...                 0.071460    0.000000
50%      13.370000     18.840000  ...                 0.080040    1.000000
75%      15.780000     21.800000  ...                 0.092080    1.000000
max      28.110000     39.280000  ...                 0.207500    1.000000

[8 rows x 31 columns]

Class distribution:
 target
0    212
1    357
Name: count, dtype: int64
```

### Step 2: Missing Values & Encoding — PASSED

**Task:** Check for missing values (none expected) and confirm all features are numeric; if any categorical columns existed, apply appropriate encoding (e.g., one‑hot).

**Coder attempts:** 1
**Reviewer verdict:** The code correctly checks for missing values, confirms all features are numeric, and skips unnecessary encoding, fulfilling the task requirements.

**Output:**
```
Shape: (569, 31)
Missing values per column:
 mean radius                0
mean texture               0
mean perimeter             0
mean area                  0
mean smoothness            0
mean compactness           0
mean concavity             0
mean concave points        0
mean symmetry              0
mean fractal dimension     0
radius error               0
texture error              0
perimeter error            0
area error                 0
smoothness error           0
compactness error          0
concavity error            0
concave points error       0
symmetry error             0
fractal dimension error    0
worst radius               0
worst texture              0
worst perimeter            0
worst area                 0
worst smoothness           0
worst compactness          0
worst concavity            0
worst concave points       0
worst symmetry             0
worst fractal dimension    0
target                     0
dtype: int64
Non-numeric columns (excluding target): []
All feature columns are numeric. No encoding needed.
Processed dataset saved to processed_data.csv
```

### Step 3: Feature Engineering/Selection — PASSED

**Task:** Create any useful derived features (e.g., ratios) and use correlation analysis or variance threshold to drop redundant features, optionally apply scaling.

**Coder attempts:** 1
**Reviewer verdict:** The code correctly creates derived features, removes highly correlated columns, applies variance threshold and scaling without using the target as a feature, and outputs the processed dataset as required.

**Output:**
```
Original shape: (569, 31)
Columns to drop due to high correlation (>0.9): ['mean perimeter', 'mean area', 'mean concave points', 'perimeter error', 'area error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst concave points', 'mean_area_perimeter_ratio', 'worst_area_perimeter_ratio', 'perimeter_error_to_mean_ratio', 'area_error_to_mean_ratio', 'smoothness_error_to_mean_ratio', 'concave_points_error_to_mean_ratio']
Columns dropped by variance threshold: []
Processed shape (features + target): (569, 28)
Total features after engineering & dropping: 27
```

### Step 4: Model Training & Comparison — PASSED

**Task:** Split data into train/test (e.g., 80/20 stratified), train at least two classifiers such as Logistic Regression and RandomForest, and record cross‑validated performance.

**Coder attempts:** 1
**Reviewer verdict:** The code correctly splits the data, avoids leakage, trains two models, reports cross‑validated and test performance, and the results are realistic for the dataset.

**Output:**
```
Full dataset shape: (569, 31)
Features shape: (569, 30), Target shape: (569,)
Train shape: (455, 30), Test shape: (114, 30)
LogisticRegression CV accuracy: 0.9780 ± 0.0098
LogisticRegression Test accuracy: 0.9825
LogisticRegression Classification Report:
              precision    recall  f1-score   support

           0     0.9762    0.9762    0.9762        42
           1     0.9861    0.9861    0.9861        72

    accuracy                         0.9825       114
   macro avg     0.9812    0.9812    0.9812       114
weighted avg     0.9825    0.9825    0.9825       114

RandomForest CV accuracy: 0.9604 ± 0.0149
RandomForest Test accuracy: 0.9474
RandomForest Classification Report:
              precision    recall  f1-score   support

           0     0.9091    0.9524    0.9302        42
           1     0.9714    0.9444    0.9577        72

    accuracy                         0.9474       114
   macro avg     0.9403    0.9484    0.9440       114
weighted avg     0.9485    0.9474    0.9476       114

Processed training data saved to 'processed_train.csv'
```

### Step 5: Evaluation & Visualization — PASSED

**Task:** Select the best model based on ROC‑AUC, compute classification report, plot ROC curve and confusion matrix, and save the figure to file.

**Coder attempts:** 1
**Reviewer verdict:** The code correctly selects the best model by ROC‑AUC, prints the classification report, plots ROC and confusion matrix, and saves the figure, with no evident data leakage.

**Output:**
```
LogisticRegression ROC‑AUC: 0.9954
RandomForest ROC‑AUC: 0.9944

Best model based on ROC‑AUC: LogisticRegression

Classification Report (best model):
              precision    recall  f1-score   support

           0       0.98      0.98      0.98        42
           1       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114


Saved evaluation figure to 'model_evaluation.png'
```

### Step 6: Model Persistence — PASSED

**Task:** Fit the best model on the full training data and serialize it to disk using joblib.

**Coder attempts:** 1
**Reviewer verdict:** The code correctly fits the model on the training data, re‑trains on the full dataset, and saves it with joblib without using any test data.

**Review history:**

- Attempt 1: REJECTED — The code fits the final model on `data.csv`, which likely contains the entire dataset (including test data), causing data leakage; the task asked to fit on the full training data only.
- Attempt 2: APPROVED — The code correctly fits the model on the training data, re‑trains on the full dataset, and saves it with joblib without using any test data.

**Output:**
```
Training data shape: (455, 30), Target distribution: [170 285]
Validation accuracy: 0.9560
Cross‑validation accuracy: 0.9582 ± 0.0224
Model saved to best_model.joblib
```

## Charts

![feature_distributions](feature_distributions.png)
![model_evaluation](model_evaluation.png)
![target_balance](target_balance.png)

In [13]:
from src.tracking import log_run_to_mlflow

run_id = log_run_to_mlflow(result_v7, ws7, goal)
print("Logged to MLflow, run_id:", run_id)

Logged to MLflow, run_id: e5b069bb380b4cefae9785d3764d3f2e


In [14]:
import os
print(sorted(os.listdir(".")))

['.env', '.gitignore', '.ipynb_checkpoints', '.pytest_cache', '.venv', 'Dockerfile.sandbox', 'mlflow.db', 'mlruns', 'notebooks', 'pytest.ini', 'requirements.txt', 'runs', 'src', 'tests']


In [15]:
from pathlib import Path
Path("api").mkdir(exist_ok=True)
Path("app").mkdir(exist_ok=True)   # while we're at it, for the Streamlit file later
print("api exists:", Path("api").exists())
print("app exists:", Path("app").exists())

api exists: True
app exists: True


In [16]:
%%writefile api/main.py
from fastapi import FastAPI, UploadFile, File, Form, BackgroundTasks, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
import uuid
import shutil
from pathlib import Path

from src.orchestrator import run_pipeline
from src.report import generate_report
from src.config import RUNS_DIR

app = FastAPI(title="Auto Data Scientist API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

RUN_STATUS = {}

Writing api/main.py


In [17]:
with open("api/main.py", encoding="utf-8") as f:
    print(f.readline())

from fastapi import FastAPI, UploadFile, File, Form, BackgroundTasks, HTTPException



In [18]:
RUN_STATUS[run_id] = {
    "status": "running",       # "running" | "success" | "failed"
    "report_path": None,       # set once the report file exists
    "error": None,             # set if something went wrong
}


NameError: name 'RUN_STATUS' is not defined

In [19]:
%%writefile api/main.py
from fastapi import FastAPI, UploadFile, File, Form, BackgroundTasks, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
import uuid
import shutil
from pathlib import Path

from src.orchestrator import run_pipeline
from src.report import generate_report
from src.config import RUNS_DIR

app = FastAPI(title="Auto Data Scientist API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# In-memory store of run status. Fine for a portfolio project;
# a real product would use a database instead, since this resets on restart.
RUN_STATUS = {}


def _execute_run(run_id: str, csv_path: Path, goal: str, workspace: Path):
    """Runs in the background. Updates RUN_STATUS when it finishes, either way."""
    try:
        result = run_pipeline(csv_path=csv_path, goal=goal, workspace=workspace, verbose=False)

        if result.success:
            report_path = generate_report(result, goal, workspace, workspace / "report.md")
            RUN_STATUS[run_id] = {
                "status": "success",
                "report_path": str(report_path),
                "error": None,
            }
        else:
            RUN_STATUS[run_id] = {
                "status": "failed",
                "report_path": None,
                "error": f"Pipeline failed at step {result.failed_at_step}",
            }

    except Exception as e:
        # Catch-all: an unexpected crash (e.g. rate limit, Docker down) shouldn't
        # leave the run stuck showing "running" forever.
        RUN_STATUS[run_id] = {
            "status": "failed",
            "report_path": None,
            "error": str(e),
        }


@app.post("/runs")
async def start_run(background_tasks: BackgroundTasks,
                    goal: str = Form(...),
                    file: UploadFile = File(...)):
    """Upload a CSV + goal. Returns immediately with a run_id; work happens in the background."""
    run_id = uuid.uuid4().hex[:12]
    workspace = RUNS_DIR / f"api_{run_id}" / "workspace"
    workspace.mkdir(parents=True, exist_ok=True)

    # Save the uploaded file to disk
    csv_path = workspace.parent / "source_data.csv"
    with open(csv_path, "wb") as f:
        shutil.copyfileobj(file.file, f)

    RUN_STATUS[run_id] = {"status": "running", "report_path": None, "error": None}

    # Schedule the actual pipeline to run after this request returns
    background_tasks.add_task(_execute_run, run_id, csv_path, goal, workspace)

    return {"run_id": run_id, "status": "running"}


@app.get("/runs/{run_id}/status")
async def get_status(run_id: str):
    """Poll this to check whether a run is still going, succeeded, or failed."""
    if run_id not in RUN_STATUS:
        raise HTTPException(status_code=404, detail="Unknown run_id")
    return RUN_STATUS[run_id]


@app.get("/runs/{run_id}/report")
async def get_report(run_id: str):
    """Download the finished Markdown report."""
    if run_id not in RUN_STATUS:
        raise HTTPException(status_code=404, detail="Unknown run_id")

    status = RUN_STATUS[run_id]
    if status["status"] != "success":
        raise HTTPException(status_code=400, detail=f"Run is not finished successfully (status: {status['status']})")

    return FileResponse(status["report_path"], media_type="text/markdown", filename="report.md")


@app.get("/health")
async def health():
    """Quick check that the API is up."""
    return {"status": "ok"}

Overwriting api/main.py


In [20]:
%%writefile api/main.py
from fastapi import FastAPI, UploadFile, File, Form, BackgroundTasks, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
import uuid
import shutil
import json
from pathlib import Path

from src.orchestrator import run_pipeline
from src.report import generate_report
from src.config import RUNS_DIR

app = FastAPI(title="Auto Data Scientist API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# In-memory store of run status. Fast, but wiped on any server restart --
# so we also write each update to disk as a fallback (see _write_status/_read_status).
RUN_STATUS = {}


def _status_file(workspace: Path) -> Path:
    return workspace / "status.json"


def _write_status(workspace: Path, status: dict):
    """Persist status to disk, so it survives a server restart."""
    with open(_status_file(workspace), "w", encoding="utf-8") as f:
        json.dump(status, f)


def _read_status(workspace: Path) -> dict | None:
    """Read status from disk if it exists (used when a run_id isn't in memory anymore)."""
    path = _status_file(workspace)
    if not path.exists():
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def _workspace_for(run_id: str) -> Path:
    return RUNS_DIR / f"api_{run_id}" / "workspace"


def _set_status(run_id: str, workspace: Path, status: dict):
    """Update both the in-memory store and the on-disk copy together."""
    RUN_STATUS[run_id] = status
    _write_status(workspace, status)


def _execute_run(run_id: str, csv_path: Path, goal: str, workspace: Path):
    """Runs in the background. Updates status when it finishes, either way."""
    try:
        result = run_pipeline(csv_path=csv_path, goal=goal, workspace=workspace, verbose=False)

        if result.success:
            report_path = generate_report(result, goal, workspace, workspace / "report.md")
            _set_status(run_id, workspace, {
                "status": "success",
                "report_path": str(report_path),
                "error": None,
            })
        else:
            _set_status(run_id, workspace, {
                "status": "failed",
                "report_path": None,
                "error": f"Pipeline failed at step {result.failed_at_step}",
            })

    except Exception as e:
        _set_status(run_id, workspace, {
            "status": "failed",
            "report_path": None,
            "error": str(e),
        })


@app.post("/runs")
async def start_run(background_tasks: BackgroundTasks,
                    goal: str = Form(...),
                    file: UploadFile = File(...)):
    """Upload a CSV + goal. Returns immediately with a run_id; work happens in the background."""
    run_id = uuid.uuid4().hex[:12]
    workspace = _workspace_for(run_id)
    workspace.mkdir(parents=True, exist_ok=True)

    csv_path = workspace.parent / "source_data.csv"
    with open(csv_path, "wb") as f:
        shutil.copyfileobj(file.file, f)

    _set_status(run_id, workspace, {"status": "running", "report_path": None, "error": None})

    background_tasks.add_task(_execute_run, run_id, csv_path, goal, workspace)

    return {"run_id": run_id, "status": "running"}


@app.get("/runs/{run_id}/status")
async def get_status(run_id: str):
    """Poll this to check whether a run is still going, succeeded, or failed.

    Checks memory first (fast); falls back to disk if the server restarted
    since the run started, so status isn't lost just because the process was.
    """
    if run_id in RUN_STATUS:
        return RUN_STATUS[run_id]

    workspace = _workspace_for(run_id)
    status = _read_status(workspace)
    if status is None:
        raise HTTPException(status_code=404, detail="Unknown run_id")

    # Recovered from disk -- also put it back in memory so future polls are fast
    RUN_STATUS[run_id] = status
    return status


@app.get("/runs/{run_id}/report")
async def get_report(run_id: str):
    """Download the finished Markdown report."""
    workspace = _workspace_for(run_id)
    status = RUN_STATUS.get(run_id) or _read_status(workspace)

    if status is None:
        raise HTTPException(status_code=404, detail="Unknown run_id")
    if status["status"] != "success":
        raise HTTPException(status_code=400, detail=f"Run is not finished successfully (status: {status['status']})")

    return FileResponse(status["report_path"], media_type="text/markdown", filename="report.md")


@app.get("/health")
async def health():
    return {"status": "ok"}

Overwriting api/main.py


In [3]:
import os
print(os.getcwd())

C:\Users\makar\Python\auto-data-scientist\notebooks


In [6]:
from pathlib import Path
import os, sys

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

print("Working directory:", Path.cwd())

Working directory: C:\Users\makar\Python\auto-data-scientist


In [7]:
%%writefile app/streamlit_app.py
import time
import requests
import streamlit as st

API_URL = "http://localhost:8000"

st.set_page_config(page_title="Auto Data Scientist", layout="centered")
st.title("🤖 Auto Data Scientist")
st.write("Upload a CSV, describe what you want predicted, and let the agents handle the rest.")

# --- Inputs ---
uploaded_file = st.file_uploader("Upload your dataset (CSV)", type=["csv"])
goal = st.text_area(
    "What do you want to predict?",
    placeholder="e.g. Predict whether a tumor is malignant or benign, using the 'target' column.",
)

start_clicked = st.button("Run analysis", type="primary", disabled=not (uploaded_file and goal))

# --- Session state: remember the run_id across reruns (Streamlit reruns the whole script on every interaction) ---
if "run_id" not in st.session_state:
    st.session_state.run_id = None

# --- Start a new run ---
if start_clicked:
    files = {"file": (uploaded_file.name, uploaded_file.getvalue(), "text/csv")}
    data = {"goal": goal}
    response = requests.post(f"{API_URL}/runs", files=files, data=data)

    if response.status_code == 200:
        st.session_state.run_id = response.json()["run_id"]
        st.success(f"Started! Run ID: {st.session_state.run_id}")
    else:
        st.error(f"Failed to start run: {response.text}")

# --- Poll and show progress for the active run ---
if st.session_state.run_id:
    run_id = st.session_state.run_id
    status_placeholder = st.empty()

    with st.spinner("Agents are working on it... this can take a few minutes."):
        while True:
            status_resp = requests.get(f"{API_URL}/runs/{run_id}/status")
            status = status_resp.json()

            if status["status"] == "running":
                status_placeholder.info("Status: running...")
                time.sleep(5)
                continue

            elif status["status"] == "success":
                status_placeholder.success("Done! Pipeline completed successfully.")
                report_resp = requests.get(f"{API_URL}/runs/{run_id}/report")
                st.download_button(
                    "Download report (Markdown)",
                    data=report_resp.content,
                    file_name="report.md",
                    mime="text/markdown",
                )
                st.markdown("---")
                st.markdown(report_resp.content.decode("utf-8"))
                break

            else:  # failed
                status_placeholder.error(f"Run failed: {status.get('error')}")
                break

Overwriting app/streamlit_app.py


In [9]:
step1_log = result_debug.logs[0]
print("Coder success:", step1_log.coder_success)
print("\n--- STDOUT ---")
print(step1_log.stdout)
print("\n--- STDERR ---")
print(step1_log.stderr)

Coder success: False

--- STDOUT ---


--- STDERR ---
failed to connect to the docker API at npipe:////./pipe/dockerDesktopLinuxEngine; check if the path is correct and if the daemon is running: open //./pipe/dockerDesktopLinuxEngine: The system cannot find the file specified.



In [10]:
   import subprocess
   result = subprocess.run(["docker", "ps"], capture_output=True, text=True)
   print("STDOUT:", result.stdout)
   print("STDERR:", result.stderr)
   print("Return code:", result.returncode)

STDOUT: CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES

STDERR: 
Return code: 0


In [11]:
result_debug2 = run_pipeline(
    csv_path=RUNS_DIR / "part4_full_run" / "source_data.csv",
    goal="Predict whether a tumor is malignant or benign, using the 'target' column.",
    workspace=RUNS_DIR / "debug_step1_v2" / "workspace",
    verbose=True,
)
print("Overall success:", result_debug2.success)

ERROR! Session/line number was not unique in database. History logging moved to new session 564
[orchestrator] Planning...
[orchestrator] Plan has 6 steps, problem_type=classification

[orchestrator] --- Step 1: Data Exploration ---
[coder] attempt 1/4: OK

[orchestrator] --- Step 2: Missing Values & Encoding ---
[coder] attempt 1/4: OK

[orchestrator] --- Step 3: Feature Engineering / Selection ---
[coder] attempt 1/4: OK
[orchestrator] Reviewer rejected (retry 1/2): The preprocessing (scaling, polynomial expansion, variance filtering, and RFE) was fit on the entire dataset before any train‑test split, leaking information from the test set into the training pipeline.
[coder] attempt 1/4: OK

[orchestrator] --- Step 4: Model Training & Comparison ---
[coder] attempt 1/4: OK

[orchestrator] --- Step 5: Best Model Evaluation & Visualization ---
[coder] attempt 1/4: OK
[orchestrator] Reviewer rejected (retry 1/2): The code selects the best model based on ROC‑AUC computed on the held‑out t

In [12]:
%%writefile api/main.py
from fastapi import FastAPI, UploadFile, File, Form, BackgroundTasks, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
import uuid
import shutil
import json
from pathlib import Path

from src.orchestrator import run_pipeline
from src.report import generate_report
from src.config import RUNS_DIR

app = FastAPI(title="Auto Data Scientist API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

RUN_STATUS = {}


def _status_file(workspace: Path) -> Path:
    return workspace / "status.json"


def _write_status(workspace: Path, status: dict):
    with open(_status_file(workspace), "w", encoding="utf-8") as f:
        json.dump(status, f)


def _read_status(workspace: Path) -> dict | None:
    path = _status_file(workspace)
    if not path.exists():
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def _workspace_for(run_id: str) -> Path:
    return RUNS_DIR / f"api_{run_id}" / "workspace"


def _set_status(run_id: str, workspace: Path, status: dict):
    RUN_STATUS[run_id] = status
    _write_status(workspace, status)


def _execute_run(run_id: str, csv_path: Path, goal: str, workspace: Path):
    """Runs in the background. Updates status when it finishes, either way."""
    try:
        result = run_pipeline(csv_path=csv_path, goal=goal, workspace=workspace, verbose=False)

        if result.success:
            report_path = generate_report(result, goal, workspace, workspace / "report.md")
            _set_status(run_id, workspace, {
                "status": "success",
                "report_path": str(report_path),
                "error": None,
            })
        else:
            # Pull the real reason from the last attempted step, instead of
            # just reporting which step number failed with no explanation.
            last_log = result.logs[-1] if result.logs else None
            if last_log is None:
                detail = "No steps were attempted."
            elif not last_log.coder_success:
                # The code itself never ran successfully -- show the raw error
                # (e.g. a Docker connection failure, a crash, etc.)
                detail = f"Code execution failed: {last_log.stderr.strip()[-500:]}"
            else:
                # The code ran, but the reviewer rejected it and retries ran out
                detail = f"Rejected by reviewer: {last_log.review_reason}"

            _set_status(run_id, workspace, {
                "status": "failed",
                "report_path": None,
                "error": f"Pipeline failed at step {result.failed_at_step} ({last_log.title if last_log else '?'}). {detail}",
            })

    except Exception as e:
        _set_status(run_id, workspace, {
            "status": "failed",
            "report_path": None,
            "error": str(e),
        })


@app.post("/runs")
async def start_run(background_tasks: BackgroundTasks,
                    goal: str = Form(...),
                    file: UploadFile = File(...)):
    run_id = uuid.uuid4().hex[:12]
    workspace = _workspace_for(run_id)
    workspace.mkdir(parents=True, exist_ok=True)

    csv_path = workspace.parent / "source_data.csv"
    with open(csv_path, "wb") as f:
        shutil.copyfileobj(file.file, f)

    _set_status(run_id, workspace, {"status": "running", "report_path": None, "error": None})

    background_tasks.add_task(_execute_run, run_id, csv_path, goal, workspace)

    return {"run_id": run_id, "status": "running"}


@app.get("/runs/{run_id}/status")
async def get_status(run_id: str):
    if run_id in RUN_STATUS:
        return RUN_STATUS[run_id]

    workspace = _workspace_for(run_id)
    status = _read_status(workspace)
    if status is None:
        raise HTTPException(status_code=404, detail="Unknown run_id")

    RUN_STATUS[run_id] = status
    return status


@app.get("/runs/{run_id}/report")
async def get_report(run_id: str):
    workspace = _workspace_for(run_id)
    status = RUN_STATUS.get(run_id) or _read_status(workspace)

    if status is None:
        raise HTTPException(status_code=404, detail="Unknown run_id")
    if status["status"] != "success":
        raise HTTPException(status_code=400, detail=f"Run is not finished successfully (status: {status['status']})")

    return FileResponse(status["report_path"], media_type="text/markdown", filename="report.md")


@app.get("/health")
async def health():
    return {"status": "ok"}

Overwriting api/main.py


In [13]:
%%writefile src/report.py
from pathlib import Path

from src.orchestrator import RunResult


def generate_report(result: RunResult, goal: str, workspace: Path, output_path: Path) -> Path:
    """Turn a RunResult into one readable Markdown file."""
    lines = [
        f"# Auto Data Scientist Report",
        f"",
        f"**Goal:** {goal}",
        f"**Problem type:** {result.plan.problem_type}",
        f"**Target column:** {result.plan.target_column}",
        f"**Overall result:** {'SUCCESS' if result.success else f'FAILED at step {result.failed_at_step}'}",
        f"",
        f"## Steps",
    ]

    for log in result.logs:
        status = "PASSED" if log.approved else "FAILED"
        lines += [
            f"",
            f"### Step {log.step_id}: {log.title} — {status}",
            f"",
            f"**Task:** {log.task}",
            f"",
            f"**Coder attempts:** {log.coder_attempts}",
            f"**Reviewer verdict:** {log.review_reason}",
        ]
        if log.leakage_flags:
            lines.append(f"**Leakage flags:** {', '.join(log.leakage_flags)}")

        if len(log.review_history) > 1:
            lines += ["", "**Review history:**", ""]
            for entry in log.review_history:
                verdict = "APPROVED" if entry["approved"] else "REJECTED"
                lines.append(f"- Attempt {entry['attempt']}: {verdict} — {entry['reason']}")

        if log.stdout.strip():
            lines += ["", "**Output:**", "```", log.stdout.strip(), "```"]

    charts = sorted(Path(workspace).glob("*.png"))
    if charts:
        lines += ["", "## Charts", ""]
        for chart in charts:
            lines.append(f"![{chart.stem}]({chart.name})")

    output_path = Path(output_path)
    output_path.write_text("\n".join(lines), encoding="utf-8")
    return output_path


def generate_pdf_report(result: RunResult, goal: str, workspace: Path, output_path: Path) -> Path:
    """Turn a RunResult into a PDF report, mirroring generate_report's structure."""
    from reportlab.lib.pagesizes import letter
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch
    from reportlab.platypus import (
        SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, PageBreak, Preformatted
    )

    output_path = Path(output_path)
    doc = SimpleDocTemplate(str(output_path), pagesize=letter,
                            topMargin=0.75 * inch, bottomMargin=0.75 * inch)
    styles = getSampleStyleSheet()
    code_style = ParagraphStyle(
        "Code", parent=styles["Normal"], fontName="Courier", fontSize=8, leading=10,
    )

    story = []

    story.append(Paragraph("Auto Data Scientist Report", styles["Title"]))
    story.append(Spacer(1, 12))
    story.append(Paragraph(f"<b>Goal:</b> {goal}", styles["Normal"]))
    story.append(Paragraph(f"<b>Problem type:</b> {result.plan.problem_type}", styles["Normal"]))
    story.append(Paragraph(f"<b>Target column:</b> {result.plan.target_column}", styles["Normal"]))
    overall = "SUCCESS" if result.success else f"FAILED at step {result.failed_at_step}"
    story.append(Paragraph(f"<b>Overall result:</b> {overall}", styles["Normal"]))
    story.append(Spacer(1, 20))

    for log in result.logs:
        status = "PASSED" if log.approved else "FAILED"
        story.append(Paragraph(f"Step {log.step_id}: {log.title} — {status}", styles["Heading2"]))
        story.append(Paragraph(f"<b>Task:</b> {log.task}", styles["Normal"]))
        story.append(Spacer(1, 4))
        story.append(Paragraph(f"<b>Coder attempts:</b> {log.coder_attempts}", styles["Normal"]))
        story.append(Paragraph(f"<b>Reviewer verdict:</b> {log.review_reason}", styles["Normal"]))

        if log.leakage_flags:
            story.append(Paragraph(f"<b>Leakage flags:</b> {', '.join(log.leakage_flags)}", styles["Normal"]))

        if len(log.review_history) > 1:
            story.append(Spacer(1, 4))
            story.append(Paragraph("<b>Review history:</b>", styles["Normal"]))
            for entry in log.review_history:
                verdict = "APPROVED" if entry["approved"] else "REJECTED"
                story.append(Paragraph(f"&bull; Attempt {entry['attempt']}: {verdict} — {entry['reason']}",
                                       styles["Normal"]))

        if log.stdout.strip():
            story.append(Spacer(1, 6))
            story.append(Paragraph("<b>Output:</b>", styles["Normal"]))
            truncated = log.stdout.strip()[:2000]
            story.append(Preformatted(truncated, code_style))

        story.append(Spacer(1, 16))

    charts = sorted(Path(workspace).glob("*.png"))
    if charts:
        story.append(PageBreak())
        story.append(Paragraph("Charts", styles["Title"]))
        story.append(Spacer(1, 12))
        for chart in charts:
            story.append(Paragraph(chart.stem, styles["Heading3"]))
            try:
                img = RLImage(str(chart), width=5.5 * inch, height=3.5 * inch, kind="proportional")
                story.append(img)
            except Exception:
                story.append(Paragraph(f"(could not embed {chart.name})", styles["Normal"]))
            story.append(Spacer(1, 16))

    doc.build(story)
    return output_path

Overwriting src/report.py


In [14]:
from src.report import generate_pdf_report

pdf_path = generate_pdf_report(result_v7, goal, ws7, ws7 / "report.pdf")
print("PDF saved to:", pdf_path)

NameError: name 'result_v7' is not defined

In [15]:
import pickle
from src.config import RUNS_DIR

ws7 = RUNS_DIR / "part4_full_run_v7" / "workspace"
with open(ws7 / "run_result.pkl", "rb") as f:
    result_v7 = pickle.load(f)

goal = "Predict whether a tumor is malignant or benign, using the 'target' column."

print("Overall success:", result_v7.success)

Overall success: True


In [17]:
%pip install reportlab

^C
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ----------------

In [18]:
from src.report import generate_pdf_report

pdf_path = generate_pdf_report(result_v7, goal, ws7, ws7 / "report.pdf")
print("PDF saved to:", pdf_path)

PDF saved to: C:\Users\makar\Python\auto-data-scientist\runs\part4_full_run_v7\workspace\report.pdf


In [19]:
%%writefile api/main.py
from fastapi import FastAPI, UploadFile, File, Form, BackgroundTasks, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
import uuid
import shutil
import json
from pathlib import Path

from src.orchestrator import run_pipeline
from src.report import generate_report, generate_pdf_report
from src.config import RUNS_DIR

app = FastAPI(title="Auto Data Scientist API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

RUN_STATUS = {}


def _status_file(workspace: Path) -> Path:
    return workspace / "status.json"


def _write_status(workspace: Path, status: dict):
    with open(_status_file(workspace), "w", encoding="utf-8") as f:
        json.dump(status, f)


def _read_status(workspace: Path) -> dict | None:
    path = _status_file(workspace)
    if not path.exists():
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def _workspace_for(run_id: str) -> Path:
    return RUNS_DIR / f"api_{run_id}" / "workspace"


def _set_status(run_id: str, workspace: Path, status: dict):

Overwriting api/main.py


In [20]:
%%writefile app/streamlit_app.py
import time
import requests
import streamlit as st

API_URL = "http://localhost:8000"

st.set_page_config(page_title="Auto Data Scientist", layout="centered")
st.title("🤖 Auto Data Scientist")
st.write("Upload a CSV, describe what you want predicted, and let the agents handle the rest.")

uploaded_file = st.file_uploader("Upload your dataset (CSV)", type=["csv"])
goal = st.text_area(
    "What do you want to predict?",
    placeholder="e.g. Predict whether a tumor is malignant or benign, using the 'target' column.",
)

start_clicked = st.button("Run analysis", type="primary", disabled=not (uploaded_file and goal))

if "run_id" not in st.session_state:
    st.session_state.run_id = None

if start_clicked:
    files = {"file": (uploaded_file.name, uploaded_file.getvalue(), "text/csv")}
    data = {"goal": goal}
    response = requests.post(f"{API_URL}/runs", files=files, data=data)

    if response.status_code == 200:
        st.session_state.run_id = response.json()["run_id"]
        st.success(f"Started! Run ID: {st.session_state.run_id}")
    else:
        st.error(f"Failed to start run: {response.text}")

if st.session_state.run_id:
    run_id = st.session_state.run_id
    status_placeholder = st.empty()

    with st.spinner("Agents are working on it... this can take a few minutes."):
        while True:
            status_resp = requests.get(f"{API_URL}/runs/{run_id}/status")
            status = status_resp.json()

            if status["status"] == "running":
                status_placeholder.info("Status: running...")
                time.sleep(5)
                continue

            elif status["status"] == "success":
                status_placeholder.success("Done! Pipeline completed successfully.")

                md_resp = requests.get(f"{API_URL}/runs/{run_id}/report")
                pdf_resp = requests.get(f"{API_URL}/runs/{run_id}/report.pdf")

                col1, col2 = st.columns(2)
                with col1:
                    st.download_button(
                        "Download report (Markdown)",
                        data=md_resp.content,
                        file_name="report.md",
                        mime="text/markdown",
                    )
                with col2:
                    st.download_button(
                        "Download report (PDF)",
                        data=pdf_resp.content,
                        file_name="report.pdf",
                        mime="application/pdf",
                    )

                st.markdown("---")
                st.markdown(md_resp.content.decode("utf-8"))
                break

            else:
                status_placeholder.error(f"Run failed: {status.get('error')}")
                break

Overwriting app/streamlit_app.py


In [21]:
with open("api/main.py", encoding="utf-8") as f:
    content = f.read()
print(len(content), "characters")
print(content[-500:])   # show the last 500 characters, to see where it cuts off

1153 characters
_status(workspace: Path, status: dict):
    with open(_status_file(workspace), "w", encoding="utf-8") as f:
        json.dump(status, f)


def _read_status(workspace: Path) -> dict | None:
    path = _status_file(workspace)
    if not path.exists():
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def _workspace_for(run_id: str) -> Path:
    return RUNS_DIR / f"api_{run_id}" / "workspace"


def _set_status(run_id: str, workspace: Path, status: dict):



In [22]:
%%writefile api/main.py
from fastapi import FastAPI, UploadFile, File, Form, BackgroundTasks, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
import uuid
import shutil
import json
from pathlib import Path

from src.orchestrator import run_pipeline
from src.report import generate_report, generate_pdf_report
from src.config import RUNS_DIR

app = FastAPI(title="Auto Data Scientist API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

RUN_STATUS = {}


def _status_file(workspace: Path) -> Path:
    return workspace / "status.json"


def _write_status(workspace: Path, status: dict):
    with open(_status_file(workspace), "w", encoding="utf-8") as f:
        json.dump(status, f)


def _read_status(workspace: Path) -> dict | None:
    path = _status_file(workspace)
    if not path.exists():
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def _workspace_for(run_id: str) -> Path:
    return RUNS_DIR / f"api_{run_id}" / "workspace"


def _set_status(run_id: str, workspace: Path, status: dict):
    RUN_STATUS[run_id] = status
    _write_status(workspace, status)


def _execute_run(run_id: str, csv_path: Path, goal: str, workspace: Path):
    try:
        result = run_pipeline(csv_path=csv_path, goal=goal, workspace=workspace, verbose=False)

        if result.success:
            report_path = generate_report(result, goal, workspace, workspace / "report.md")
            pdf_path = generate_pdf_report(result, goal, workspace, workspace / "report.pdf")
            _set_status(run_id, workspace, {
                "status": "success",
                "report_path": str(report_path),
                "pdf_report_path": str(pdf_path),
                "error": None,
            })
        else:
            last_log = result.logs[-1] if result.logs else None
            if last_log is None:
                detail = "No steps were attempted."
            elif not last_log.coder_success:
                detail = f"Code execution failed: {last_log.stderr.strip()[-500:]}"
            else:
                detail = f"Rejected by reviewer: {last_log.review_reason}"

            _set_status(run_id, workspace, {
                "status": "failed",
                "report_path": None,
                "pdf_report_path": None,
                "error": f"Pipeline failed at step {result.failed_at_step} ({last_log.title if last_log else '?'}). {detail}",
            })

    except Exception as e:
        _set_status(run_id, workspace, {
            "status": "failed",
            "report_path": None,
            "pdf_report_path": None,
            "error": str(e),
        })


@app.post("/runs")
async def start_run(background_tasks: BackgroundTasks,
                    goal: str = Form(...),
                    file: UploadFile = File(...)):
    run_id = uuid.uuid4().hex[:12]
    workspace = _workspace_for(run_id)
    workspace.mkdir(parents=True, exist_ok=True)

    csv_path = workspace.parent / "source_data.csv"
    with open(csv_path, "wb") as f:
        shutil.copyfileobj(file.file, f)

    _set_status(run_id, workspace, {"status": "running", "report_path": None, "pdf_report_path": None, "error": None})

    background_tasks.add_task(_execute_run, run_id, csv_path, goal, workspace)

    return {"run_id": run_id, "status": "running"}


@app.get("/runs/{run_id}/status")
async def get_status(run_id: str):
    if run_id in RUN_STATUS:
        return RUN_STATUS[run_id]

    workspace = _workspace_for(run_id)
    status = _read_status(workspace)
    if status is None:
        raise HTTPException(status_code=404, detail="Unknown run_id")

    RUN_STATUS[run_id] = status
    return status


@app.get("/runs/{run_id}/report")
async def get_report(run_id: str):
    workspace = _workspace_for(run_id)
    status = RUN_STATUS.get(run_id) or _read_status(workspace)

    if status is None:
        raise HTTPException(status_code=404, detail="Unknown run_id")
    if status["status"] != "success":
        raise HTTPException(status_code=400, detail=f"Run is not finished successfully (status: {status['status']})")

    return FileResponse(status["report_path"], media_type="text/markdown", filename="report.md")


@app.get("/runs/{run_id}/report.pdf")
async def get_pdf_report(run_id: str):
    workspace = _workspace_for(run_id)
    status = RUN_STATUS.get(run_id) or _read_status(workspace)

    if status is None:
        raise HTTPException(status_code=404, detail="Unknown run_id")
    if status["status"] != "success":
        raise HTTPException(status_code=400, detail=f"Run is not finished successfully (status: {status['status']})")

    return FileResponse(status["pdf_report_path"], media_type="application/pdf", filename="report.pdf")


@app.get("/health")
async def health():
    return {"status": "ok"}

Overwriting api/main.py


In [23]:
with open("api/main.py", encoding="utf-8") as f:
    content = f.read()
print(len(content), "characters")
print("Ends with:", repr(content[-80:]))

4900 characters
Ends with: 'ort.pdf")\n\n\n@app.get("/health")\nasync def health():\n    return {"status": "ok"}\n'


In [24]:
import ast
try:
    ast.parse(content)
    print("Valid Python syntax ✓")
except SyntaxError as e:
    print("Syntax error:", e)

Valid Python syntax ✓
